# 第二十二课:BPE 分词器——模型读到的不是字,是 token

讲义:[22-bpe.md](22-bpe.md)。本课在《西游记》全本上训练一个 byte-level BPE 分词器,并实读本地 Qwen3-4B 的 tokenizer.json 做对照。

In [ ]:
import json
import re
import time
from collections import Counter, defaultdict


# ---------- 1. byte→unicode 映射(GPT-2 同款):让 256 个字节全部可读 ----------
def bytes_to_unicode():
    """可打印字节原样保留;其余映射到 256+ 的码位——避开控制字符,词表可存 JSON。"""
    bs = (list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256)))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b); cs.append(256 + n); n += 1
    return {b: chr(c) for b, c in zip(bs, cs)}


B2U = bytes_to_unicode()          # 字节 -> 可读符号
# 「，」的三个字节在符号世界长什么样:
print([B2U[b] for b in "，".encode("utf-8")])

## 2. 预切分 + 手撕训练:增量式 BPE(讲义 §4)

In [ ]:
# 预切分:汉字串 / 拉丁字母数字串 / 空白串 / 其他单字符(切词边界,合并不越界)
PRE_TOK = re.compile(r"[\u3400-\u9fff]+|[0-9A-Za-z]+|\s+|\S")
print(PRE_TOK.findall("那猴王信受奉行，日日在洞中修炼。行者道：师父！"))

In [ ]:
def train_bpe(text, vocab_size):
    """返回 merges:第 i 项是第 i 次合并的 (a, b)。
    id 约定:0..255 = 单字节,256+i = 第 i 次合并的产物。"""
    assert vocab_size >= 256
    word_counts = Counter(PRE_TOK.findall(text))
    words = [(tuple(B2U[b] for b in w.encode("utf-8")), c)   # 词 -> 符号元组,带频次
             for w, c in word_counts.items()]

    pair_counts = Counter()               # 相邻对 -> 总频次
    pair_to_words = defaultdict(set)      # 相邻对 -> 含它的词下标(增量索引)
    for i, (syms, c) in enumerate(words):
        for p in zip(syms, syms[1:]):
            pair_counts[p] += c
            pair_to_words[p].add(i)

    merges = []
    while len(merges) < vocab_size - 256:
        # max 遇平频取先插入者(dict 保序)——真实实现的「先见先得」
        best = max(pair_counts, key=pair_counts.get)
        if pair_counts[best] < 2:          # 没有成对的频次了,提前收工
            break
        a, b = best
        merges.append(best)
        for i in list(pair_to_words[best]):      # 只重扫受影响的词 ← 增量的关键
            syms, c = words[i]
            j, new = 0, []
            while j < len(syms):
                if j < len(syms) - 1 and (syms[j], syms[j + 1]) == best:
                    new.append(a + b)            # 合并!
                    j += 2
                else:
                    new.append(syms[j]); j += 1
            new = tuple(new)
            if new == syms:
                continue
            for p in zip(syms, syms[1:]):        # 撤旧对的账
                pair_counts[p] -= c
                if not pair_counts[p]: del pair_counts[p]
                pair_to_words[p].discard(i)
            for p in zip(new, new[1:]):          # 记新对的账
                pair_counts[p] += c
                pair_to_words[p].add(i)
            words[i] = (new, c)
        pair_counts.pop(best, None)
        pair_to_words.pop(best, None)
    return merges

## 3. 手撕编码 / 解码(讲义 §5)

In [ ]:
class BPETokenizer:
    def __init__(self, merges):
        self.merges = merges
        self.rank = {m: i for i, m in enumerate(merges)}
        self.sym_to_id = {B2U[b]: b for b in range(256)}
        sym_to_bytes = {B2U[b]: bytes([b]) for b in range(256)}
        for i, (a, b) in enumerate(merges):
            self.sym_to_id[a + b] = 256 + i
            sym_to_bytes[a + b] = sym_to_bytes[a] + sym_to_bytes[b]
        self.sym_to_bytes = sym_to_bytes
        self.id_to_sym = {i: s for s, i in self.sym_to_id.items()}

    def _merge_word(self, syms):
        """按训练顺序(rank)贪心合并——顺序与训练一致,结果才与词表对齐。"""
        while len(syms) > 1:
            best_k, best_rank = None, None
            for k, p in enumerate(zip(syms, syms[1:])):
                r = self.rank.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best_k, best_rank = k, r
            if best_k is None:
                return syms
            syms[best_k:best_k + 2] = [syms[best_k] + syms[best_k + 1]]
        return syms

    def encode(self, text):
        ids = []
        for w in PRE_TOK.findall(text):
            syms = [B2U[b] for b in w.encode("utf-8")]
            for s in self._merge_word(syms):
                ids.append(self.sym_to_id[s])
        return ids

    def decode(self, ids):
        raw = b"".join(self.sym_to_bytes[self.id_to_sym[i]] for i in ids)
        return raw.decode("utf-8", errors="replace")

    def id_to_token(self, i):
        return self.sym_to_bytes[self.id_to_sym[i]].decode("utf-8", errors="replace")

## 4. 实验现场(讲义 §6–8 的数字全部来自这里)

In [ ]:
text = open("data/xiyouji.txt", encoding="utf-8").read()
print(f"语料:《西游记》{len(text):,} 字符,{len(text.encode('utf-8')):,} 字节,唯一字符 {len(set(text)):,}")

print("\n[实验1] 词表大小 vs 压缩率(全语料,真实运行):")
for vs in (512, 1024, 2048, 4096):
    t0 = time.time()
    m = train_bpe(text, vocab_size=vs)
    tok = BPETokenizer(m)
    ids = tok.encode(text)
    ok = tok.decode(ids) == text
    print(f"  词表 {vs:5d}: {time.time()-t0:5.1f}s | {len(ids):,} token | "
          f"{len(text.encode('utf-8'))/len(ids):.2f} 字节/token | "
          f"字级对照 {len(text):,} token | 往返无损 {ok}")

In [ ]:
print("[实验2] 主角登场:词表 2048")
t0 = time.time()
merges = train_bpe(text, vocab_size=2048)
tok = BPETokenizer(merges)
print(f"  训练 {time.time()-t0:.1f}s,学到 {len(merges)} 个合并")
sample = "那猴王信受奉行，日日在洞中修炼。行者道：师父，八戒又在偷懒！"
ids = tok.encode(sample)
print(f"  「{sample}」")
print(f"  {len(sample)} 字 → {len(ids)} token: {[tok.id_to_token(i) for i in ids]}")
print(f"  懒 在语料中出现 {text.count('懒')} 次——频次不够铸成整字铅字,"
      f"退回字节级合并片段;无损依旧(往返 {tok.decode(ids) == sample})")
first_cjk = next(i for i, (a, b) in enumerate(merges)
                 if len(tok.sym_to_bytes[a+b]) >= 3
                 and '\u4e00' <= tok.id_to_token(256+i)[:1] <= '\u9fff')
print(f"  第 {first_cjk+1} 次合并才铸出第一个整汉字:'{tok.id_to_token(256+first_cjk)}'"
      f"(此前都在拼高频字的 UTF-8 字节前缀)")
print("  合并 #500 附近(已是词的世界):",
      [tok.id_to_token(256+i) for i in range(495, 505)])
print("  西游记词汇是否铸成了单 token:")
for w in ["行者", "八戒", "唐僧", "菩萨", "沙僧", "悟空", "妖精", "孙悟空"]:
    print(f"    {w}: {len(tok.encode(w))} token")

In [ ]:
print("[实验3] 对照 Qwen3-4B(本地 tokenizer.json 实读):")
QWEN = ("/Volumes/Sam-APFS/huggingface/hub/models--mlx-community--"
        "Qwen3-4B-Instruct-2507-bf16/snapshots/"
        "f9e77d4283734966e9cd641bf35547f0cff5d427")
tj = json.load(open(f"{QWEN}/tokenizer.json", encoding="utf-8"))
qvocab = tj["model"]["vocab"]
u2b = {s: b for b, s in B2U.items()}
def q_detok(s):
    try:
        return bytes(u2b[ch] for ch in s).decode("utf-8")
    except Exception:
        return None
qset = {}
for k in qvocab:
    d = q_detok(k)
    if d is not None:
        qset[d] = qvocab[k]
print(f"  基础词表 {len(qvocab):,} 条 + 特殊 token {len(tj['added_tokens'])} 条,"
      f" merges {len(tj['model']['merges']):,} 条")
for w in ["行者", "八戒", "唐僧", "菩萨", "悟空", "进行", "银行", "行政"]:
    print(f"    '{w}' 是 Qwen 单 token: {w in qset}")
xing = [t for t in qset if "行" in t]
print(f"  含'行'的 Qwen token {len(xing)} 个,样例: {sorted(x for x in xing if len(x) <= 3)[:10]}")
n_cjk = sum(1 for t in qset if any("\u4e00" <= ch <= "\u9fff" for ch in t))
print(f"  Qwen 词表中含汉字的 token:{n_cjk:,} 条")

In [ ]:
print("[实验4] 词表大小的参数账(Qwen3-4B:config.json 实读):")
cfg = json.load(open(f"{QWEN}/config.json", encoding="utf-8"))
h, v = cfg["hidden_size"], cfg["vocab_size"]
print(f"  hidden_size={h}, vocab_size={v}")
print(f"  embedding + lm_head = 2×{v}×{h} = {2*v*h:,} 参数"
      f"(约 {2*v*h/1e6:.0f}M,占 4.02B 总参数的 {2*v*h/4.02e9:.0%})")
print("  词表换序列长度(实验1 的真实汇率):词表 512→4096(×8),"
      "序列 1,094,677→602,897 token(−45%)")

In [ ]:
# ---- 作业现场 ----

# TODO(C1): 默写增量更新;量一下全量重扫版在小样本上的耗时差

# TODO(C2): 把 _merge_word 改成「任选可合并对」,构造切分不一致的反例

# TODO(C3): round-trip 压力测试:随机抽 100 段,断言 decode(encode(x)) == x
# import random
# rng = random.Random(0)
# pos = [rng.randrange(len(text) - 200) for _ in range(100)]
# assert all(tok.decode(tok.encode(text[p:p+200])) == text[p:p+200] for p in pos)

# TODO(D1): 自训 32K 词表、d=4096:2×32768×4096 参数,bf16 多大?

# TODO(E1): vocab_size=8192 再训一次,压缩率?对照实验1 画边际收益

# TODO(E2/预习): 下一课把 Mini-GPT 的反向传播焊出来;先想:embedding 的梯度
#                和矩阵的梯度(第12课两模板)在这里分别长什么样?
